# Notebook 2 — Train the passive deepfake detectors (Xception, EfficientNet-B4)

**What it does:** trains a real/fake classifier on FF++ faces, then tests it on
FF++ (same data) and Celeb-DF (unseen data).

**Kaggle settings:** Input: your `deepfake-faces` dataset (output of notebook 1) ·
Accelerator: **GPU P100** · Internet: **On** (downloads ImageNet weights)

To run both models in parallel on two accounts, set `RUN = ["xception"]` on one
and `RUN = ["effb4"]` on the other.

**Output:** `det_<model>.pt`, `results_detectors.csv`, `results_per_method.csv`,
`preds_<model>.csv`

In [ ]:
import os, glob, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import timm
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm.auto import tqdm

RUN = ["xception", "effb4"]
MODELS = {   # several names tried in order, because timm renames models between versions
    "xception": ["legacy_xception.tf_in1k", "legacy_xception", "xception"],
    "effb4": ["tf_efficientnet_b4.ns_jft_in1k", "tf_efficientnet_b4_ns", "efficientnet_b4"],
}
IMG, BATCH, EPOCHS, LR = 256, 32, 8, 1e-4
OUTDIR = "/kaggle/working"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda"
print("timm", timm.__version__)

meta_path = glob.glob("/kaggle/input/**/metadata.csv", recursive=True)
assert meta_path, "metadata.csv not found - add the deepfake-faces dataset as Input"
ROOT = os.path.dirname(meta_path[0])
meta = pd.read_csv(meta_path[0])
print(meta.groupby(["dataset", "split", "label"]).size().unstack(fill_value=0))

tr = meta[(meta.dataset == "ffpp") & (meta.split == "train")].reset_index(drop=True)
va = meta[(meta.dataset == "ffpp") & (meta.split == "val")].reset_index(drop=True)
te = meta[(meta.dataset == "ffpp") & (meta.split == "test")].reset_index(drop=True)
cd = meta[meta.dataset == "celebdf"].reset_index(drop=True)

In [ ]:
class FaceDS(Dataset):
    def __init__(self, df, tf):
        self.df, self.tf = df, tf
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(os.path.join(ROOT, r.path)).convert("RGB")
        return self.tf(img), torch.tensor(float(r.label))

def make_model(key):
    for name in MODELS[key]:
        try:
            return timm.create_model(name, pretrained=True, num_classes=1), name
        except Exception as e:
            print(f"  could not load {name}: {str(e)[:80]}")
    raise RuntimeError(f"No working timm name for {key}")

def get_tfs(model):
    cfg = timm.data.resolve_data_config({}, model=model)
    norm = T.Normalize(cfg["mean"], cfg["std"])
    train_tf = T.Compose([T.Resize((IMG, IMG)), T.RandomHorizontalFlip(),
                          T.ColorJitter(0.1, 0.1, 0.1), T.ToTensor(), norm])
    test_tf = T.Compose([T.Resize((IMG, IMG)), T.ToTensor(), norm])
    return train_tf, test_tf

@torch.no_grad()
def predict(model, df, tf):
    model.eval()
    dl = DataLoader(FaceDS(df, tf), batch_size=64, shuffle=False, num_workers=4)
    probs = []
    for x, _ in dl:
        with torch.autocast("cuda", dtype=torch.float16):
            probs.append(torch.sigmoid(model(x.to(device)).float().squeeze(1)).cpu())
    out = df.copy()
    out["prob"] = torch.cat(probs).numpy()
    return out

def metrics(p):
    auc = roc_auc_score(p.label, p.prob)
    acc = ((p.prob > 0.5).astype(int) == p.label).mean()
    fpr, tpr, _ = roc_curve(p.label, p.prob)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    eer = (fpr[i] + fnr[i]) / 2
    v = p.groupby("video").agg(label=("label", "first"), prob=("prob", "mean"))
    vauc = roc_auc_score(v.label, v.prob)
    return dict(frame_auc=auc, frame_acc=acc, eer=eer, video_auc=vauc)

In [ ]:
results, per_method = [], []
for key in RUN:
    print(f"\n========== {key} ==========")
    model, name = make_model(key)
    model = model.to(device)
    train_tf, test_tf = get_tfs(model)
    dl = DataLoader(FaceDS(tr, train_tf), batch_size=BATCH, shuffle=True,
                    num_workers=4, drop_last=True, pin_memory=True)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * len(dl))
    scaler = torch.cuda.amp.GradScaler()
    lossf = nn.BCEWithLogitsLoss()
    best = -1
    for ep in range(EPOCHS):
        model.train()
        pbar = tqdm(dl, desc=f"{key} epoch {ep + 1}/{EPOCHS}")
        for x, y in pbar:
            x, y = x.to(device, non_blocking=True), y.to(device)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = lossf(model(x).squeeze(1).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            pbar.set_postfix(loss=f"{loss.item():.3f}")
        val_auc = metrics(predict(model, va, test_tf))["frame_auc"]
        print(f"  val AUC = {val_auc:.4f}")
        if val_auc > best:
            best = val_auc
            torch.save({"state": model.state_dict(), "timm_name": name, "img": IMG},
                       f"{OUTDIR}/det_{key}.pt")

    model.load_state_dict(torch.load(f"{OUTDIR}/det_{key}.pt")["state"])
    p_te, p_cd = predict(model, te, test_tf), predict(model, cd, test_tf)
    pd.concat([p_te, p_cd]).to_csv(f"{OUTDIR}/preds_{key}.csv", index=False)
    for test_name, p in [("FF++ test", p_te), ("Celeb-DF (cross)", p_cd)]:
        results.append(dict(model=key, test=test_name, **metrics(p)))
    for meth in sorted(p_te.method.unique()):
        if meth == "original":
            continue
        sub = p_te[(p_te.label == 0) | (p_te.method == meth)]
        per_method.append(dict(model=key, method=meth, frame_auc=roc_auc_score(sub.label, sub.prob)))
    del model; torch.cuda.empty_cache()

res = pd.DataFrame(results)
res.to_csv(f"{OUTDIR}/results_detectors.csv", index=False)
pm = pd.DataFrame(per_method).pivot(index="model", columns="method", values="frame_auc")
pm.to_csv(f"{OUTDIR}/results_per_method.csv")
print(res.round(4).to_string(index=False))
print(pm.round(4))